# MNIST MLP3 — SGD + momentum + Muon baseline

Clean baseline only: no trace-log projection, adaptive ECS correction, WW-PGD, or spectral-flow intervention. Every epoch, including epoch 0, measures full train/test loss and accuracy plus the original WeightWatcher full-$M$ diagnostics.

$$
m_{\mathrm{mid}}=\left\lfloor\frac{m_{\mathrm{detX}}+m_{\mathrm{PL}}}{2}\right\rfloor.
$$

In [ ]:
from pathlib import Path
import importlib,subprocess,sys
try: importlib.import_module("weightwatcher")
except ImportError: subprocess.check_call([sys.executable,"-m","pip","install","-q","weightwatcher>=0.7.7"])
ROOT=None
for p in [Path.cwd(),*Path.cwd().parents]:
    q=p/"baseline"
    if (q/"rg_baselines").is_dir(): ROOT=q; break
    if (p/"rg_baselines").is_dir(): ROOT=p; break
if ROOT is None: raise RuntimeError("Run from a clone of CalculatedContent/rg_optimizers")
if str(ROOT) not in sys.path: sys.path.insert(0,str(ROOT))
print("baseline root:",ROOT)

In [ ]:
from rg_baselines import BaselineConfig,run_baseline,plot_all
from IPython.display import display
CONFIG=BaselineConfig(optimizer="sgd_momentum_muon",epochs=20,train_eval_max_batches=None,muon_parameter_names=("fc1.weight","fc2.weight"),muon_learning_rate=0.02,muon_momentum=0.95,muon_nesterov=True,muon_newton_schulz_steps=5,muon_aux_learning_rate=0.05,muon_aux_momentum=0.9)
RUN_DIR=ROOT/"runs"/"sgd_momentum_muon"
result=run_baseline(CONFIG,data_dir=ROOT/"data",output_dir=RUN_DIR,progress=True)
plot_all(result,output_dir=RUN_DIR/"plots",show=True)
print("saved:",RUN_DIR.resolve())

In [ ]:
display(result.performance)
required=["run","epoch","layer","alpha","detX_num","num_pl_spikes","ERG_gap","m_midpoint","trace_log_midpoint_total","trace_log_midpoint_per_eval","geometric_mean_midpoint","stable_rank","participation_ratio","midpoint_energy_fraction","status"]
display(result.spectral_metrics[required].sort_values(["epoch","layer"]))
display(result.optimizer_groups)